# Verificação de Idempotência — Camada Silver

**Tech Challenge Fase 2 — POSTECH AI Scientist**

Valida que o script de preparação da camada **Silver** pode ser executado múltiplas vezes sem alterar o estado final do data lake — propriedade essencial para reprocessos e retries seguros em produção.

---

**Por que idempotência importa?** Numa pipeline orquestrada (Step Functions, retries automáticos, backfills manuais), qualquer job pode ser executado mais de uma vez sobre o mesmo insumo — por falha transitória, reprocesso ou operação humana. Um job **idempotente** produz exatamente o mesmo estado final do lake em todas as execuções: sem duplicar registros, sem apagar histórico, sem variar conteúdo.

**Como o job Glue garante isso?** Escrita Parquet com `mode("overwrite")` + `partitionBy("ano")` + `spark.sql.sources.partitionOverwriteMode=dynamic`: cada execução sobrescreve **apenas as partições `ano=YYYY` presentes no lote**, deixando as demais intactas.

**Como este notebook verifica?** Usamos a simulação local fiel da pipeline (`pipeline_local.py`, pandas), executamos a camada **duas vezes** e comparamos *snapshots* do lake — uma linha por partição física com contagem de linhas e hash MD5 do conteúdo de negócio (colunas voláteis de timestamp excluídas, pois mudam legitimamente a cada execução). Idempotência confirmada = mesmas partições, mesmas contagens, mesmos hashes.

In [1]:
# Simulação local fiel dos jobs Glue (mesma lógica de transformação,
# DQ e particionamento Hive-style por ano) — ver notebooks/pipeline_local.py
import tempfile
from pathlib import Path

import pandas as pd

from pipeline_local import (
    run_bronze, run_silver, run_gold,
    snapshot_camada, comparar_snapshots,
)

DADOS_DIR = Path('../dados')
LAKE_DIR = Path(tempfile.mkdtemp(prefix='lake_'))
print(f'Data lake local desta verificação: {LAKE_DIR}')

Data lake local desta verificação: /sessions/gallant-youthful-faraday/tmp/lake_6c0y243x


## Preparação das camadas anteriores

A verificação isola a camada em teste: as camadas a montante são materializadas **uma única vez** e permanecem constantes durante o experimento.

In [2]:
# Bronze materializada uma única vez — insumo constante do experimento
run_bronze(DADOS_DIR, LAKE_DIR)

{'indicador_municipio': 23995,
 'indicador_uf': 145,
 'meta_brasil': 3,
 'meta_uf': 54,
 'meta_municipio': 10704}

## Execução 1 — estado inicial da camada Silver

In [3]:
resultado_exec1 = run_silver(LAKE_DIR)
print('Registros processados na execução 1:')
resultado_exec1

Registros processados na execução 1:


{'indicador_municipio': (23995, 0),
 'indicador_uf': (145, 0),
 'meta_brasil': (3, 0),
 'meta_uf': (54, 0),
 'meta_municipio': (10704, 0)}

In [4]:
def listar_particoes(base):
    """Lista a estrutura física de partições gravada no lake local."""
    for p in sorted((LAKE_DIR / base).rglob('ano=*')):
        n = len(pd.read_parquet(p / 'dados.parquet'))
        print(f'{p.relative_to(LAKE_DIR)}  ({n} linhas)')

listar_particoes('sot')

sot/pass/indicador_municipio/ano=2023  (11547 linhas)
sot/pass/indicador_municipio/ano=2024  (12448 linhas)
sot/pass/indicador_uf/ano=2023  (70 linhas)
sot/pass/indicador_uf/ano=2024  (75 linhas)
sot/pass/meta_brasil/ano=2023  (1 linhas)
sot/pass/meta_brasil/ano=2024  (1 linhas)
sot/pass/meta_brasil/ano=2025  (1 linhas)
sot/pass/meta_municipio/ano=2023  (5352 linhas)
sot/pass/meta_municipio/ano=2024  (5352 linhas)
sot/pass/meta_uf/ano=2023  (27 linhas)
sot/pass/meta_uf/ano=2024  (27 linhas)


In [5]:
snap_exec1 = snapshot_camada(LAKE_DIR, 'sot')
snap_exec1

,particao,n_linhas,hash_negocio
0,sot/pass/indicador_municipio/ano=2023,11547,f1361b6f916e3079fa7a8f5881c903b4
1,sot/pass/indicador_municipio/ano=2024,12448,b5a20db4b4a1b890a2a250e22219df27
2,sot/pass/indicador_uf/ano=2023,70,3b7f74fb4c7a9075ca6597ccada4d085
3,sot/pass/indicador_uf/ano=2024,75,bea479a73fd0d5f83dce578120cbb29f
4,sot/pass/meta_brasil/ano=2023,1,bee18e1652b27b8a6aac7c3ca588b2e6
5,sot/pass/meta_brasil/ano=2024,1,1dee639b049074a76079bab9403d3b3e
6,sot/pass/meta_brasil/ano=2025,1,51ba56a9e77fa9a0134fabf0af81f393
7,sot/pass/meta_municipio/ano=2023,5352,e2aa6f9f9e6684176cb5f2df09fd465a
8,sot/pass/meta_municipio/ano=2024,5352,aa83d5a8d0cff3fa9e61e6e653958936
9,sot/pass/meta_uf/ano=2023,27,5545f36337e9ab3a8a3bb9d252d5b0f4


## Execução 2 — reprocesso completo da camada Silver

Repetimos a execução sobre o mesmo insumo, simulando um retry do orquestrador ou um reprocesso manual.

In [6]:
resultado_exec2 = run_silver(LAKE_DIR)
snap_exec2 = snapshot_camada(LAKE_DIR, 'sot')
print('Registros processados na execução 2:')
resultado_exec2

Registros processados na execução 2:


{'indicador_municipio': (23995, 0),
 'indicador_uf': (145, 0),
 'meta_brasil': (3, 0),
 'meta_uf': (54, 0),
 'meta_municipio': (10704, 0)}

## Comparação dos snapshots

Partição a partição: mesma estrutura, mesma contagem e mesmo hash de conteúdo de negócio.

In [7]:
comparacao = comparar_snapshots(snap_exec1, snap_exec2)
comparacao

,particao,n_linhas_exec1,hash_negocio_exec1,n_linhas_exec2,hash_negocio_exec2,idempotente
0,sot/pass/indicador_municipio/ano=2023,11547,f1361b6f916e3079fa7a8f5881c903b4,11547,f1361b6f916e3079fa7a8f5881c903b4,True
1,sot/pass/indicador_municipio/ano=2024,12448,b5a20db4b4a1b890a2a250e22219df27,12448,b5a20db4b4a1b890a2a250e22219df27,True
2,sot/pass/indicador_uf/ano=2023,70,3b7f74fb4c7a9075ca6597ccada4d085,70,3b7f74fb4c7a9075ca6597ccada4d085,True
3,sot/pass/indicador_uf/ano=2024,75,bea479a73fd0d5f83dce578120cbb29f,75,bea479a73fd0d5f83dce578120cbb29f,True
4,sot/pass/meta_brasil/ano=2023,1,bee18e1652b27b8a6aac7c3ca588b2e6,1,bee18e1652b27b8a6aac7c3ca588b2e6,True
5,sot/pass/meta_brasil/ano=2024,1,1dee639b049074a76079bab9403d3b3e,1,1dee639b049074a76079bab9403d3b3e,True
6,sot/pass/meta_brasil/ano=2025,1,51ba56a9e77fa9a0134fabf0af81f393,1,51ba56a9e77fa9a0134fabf0af81f393,True
7,sot/pass/meta_municipio/ano=2023,5352,e2aa6f9f9e6684176cb5f2df09fd465a,5352,e2aa6f9f9e6684176cb5f2df09fd465a,True
8,sot/pass/meta_municipio/ano=2024,5352,aa83d5a8d0cff3fa9e61e6e653958936,5352,aa83d5a8d0cff3fa9e61e6e653958936,True
9,sot/pass/meta_uf/ano=2023,27,5545f36337e9ab3a8a3bb9d252d5b0f4,27,5545f36337e9ab3a8a3bb9d252d5b0f4,True


In [8]:
# Validação formal: qualquer divergência interrompe o notebook com erro
assert comparacao['idempotente'].all(), 'FALHA: execuções produziram estados diferentes!'
assert len(snap_exec1) == len(snap_exec2), 'FALHA: número de partições divergente!'

print('=' * 60)
print(f'  IDEMPOTÊNCIA CONFIRMADA — {len(comparacao)} partições idênticas')
print(f'  nas duas execuções (estrutura, contagens e conteúdo).')
print('=' * 60)

  IDEMPOTÊNCIA CONFIRMADA — 11 partições idênticas
  nas duas execuções (estrutura, contagens e conteúdo).


## Inspeção do roteamento pass/quarentena

O Silver separa registros aprovados (`sot/pass/`, particionado por ano) dos reprovados (`sot/quarentena/`, particionado pela data de processamento — trilha de auditoria, já que o próprio `ano` pode ser o campo inválido).

In [9]:
print('Roteamento por entidade (pass, quarentena):')
for entidade, (n_pass, n_quar) in resultado_exec2.items():
    total = n_pass + n_quar
    pct = n_pass / total * 100 if total else 0
    print(f'  {entidade:<22}: pass={n_pass:>6} | quarentena={n_quar:>4} | aprovação={pct:.1f}%')

Roteamento por entidade (pass, quarentena):
  indicador_municipio   : pass= 23995 | quarentena=   0 | aprovação=100.0%
  indicador_uf          : pass=   145 | quarentena=   0 | aprovação=100.0%
  meta_brasil           : pass=     3 | quarentena=   0 | aprovação=100.0%
  meta_uf               : pass=    54 | quarentena=   0 | aprovação=100.0%
  meta_municipio        : pass= 10704 | quarentena=   0 | aprovação=100.0%


## Conclusão

A camada **Silver** é idempotente: as transformações (decode de `rede`, arredondamentos, normalização de chaves) e as regras DQ (`_dq_*` → `_dq_passou`) são **determinísticas** — o mesmo registro de entrada é sempre aprovado ou reprovado pelo mesmo motivo, e as partições `sot/pass/<entidade>/ano=YYYY/` resultantes são idênticas entre execuções.

A quarentena também se mantém estável dentro de um mesmo dia de processamento (partição `anomesdia=`): reprocessos no mesmo dia sobrescrevem a mesma partição de auditoria. No ambiente AWS, o `etl_silver.py` aplica exatamente o mesmo mecanismo de overwrite dinâmico por partição.